# Notebook 03 — Baseline & CNN-LSTM Training

End-to-end demonstration of the AQI model training pipeline:

1. Build (or load) the training dataset
2. Train **Random Forest** and **Gradient Boosting** with time-split CV
3. Simple **hyperparameter grid search** for the best baseline
4. Train the **CNN-LSTM** spatio-temporal model on synthetic grids
5. Plot **loss curves**, **scatter plots**, and **feature importances**

> All cells run on synthetic data; swap in the real CSVs from `data/processed/`
> once downloads are complete.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import json
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV
import torch
import torch.nn as nn

from src.models.cnn_lstm_aqi import CNNLSTM, AQIDataset
from src.models.train_aqi import train_epoch, val_epoch

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Build / Load Training Dataset

We load `data/processed/aqi_training_dataset.csv` if available, otherwise
generate a synthetic station-level dataset.

In [ ]:
FEATURE_COLS = [
    'no2_column', 'so2_column', 'co_column', 'o3_column',
    'hcho_column', 'insat_aod',
    't2m', 'rh2m', 'u10', 'v10', 'tp', 'sp', 'blh',
]
TARGET_COLS = ['pm25_target', 'aqi_target']

csv_path = Path('../data/processed/aqi_training_dataset.csv')

if csv_path.exists():
    df = pd.read_csv(csv_path, parse_dates=['date'])
    print(f'✓ Loaded real training data: {df.shape}')
else:
    print('Generating synthetic training data…')
    rng = np.random.default_rng(0)
    n = 8000
    dates = pd.date_range('2019-01-01', periods=n, freq='D')
    dates = dates[dates.date < pd.Timestamp('2023-01-01').date()]
    n = len(dates)

    X_raw = rng.standard_normal((n, len(FEATURE_COLS)))
    # Seasonal modulation on t2m index
    t2m_idx = FEATURE_COLS.index('t2m')
    doy = np.array([d.dayofyear for d in dates])
    X_raw[:, t2m_idx] += 2 * np.sin(2 * np.pi * doy / 365)

    # pm25 ~ 0.3*no2 + 0.25*aod - 0.2*blh + noise
    no2_idx = FEATURE_COLS.index('no2_column')
    aod_idx = FEATURE_COLS.index('insat_aod')
    blh_idx = FEATURE_COLS.index('blh')
    pm25 = (
        40 + 15*X_raw[:, no2_idx] + 12*X_raw[:, aod_idx]
        - 8*X_raw[:, blh_idx] + rng.normal(0, 10, n)
    )
    pm25 = np.clip(pm25, 5, 400)
    aqi  = np.clip(pm25 * 1.1 + rng.normal(0, 5, n), 10, 500)

    df = pd.DataFrame(X_raw, columns=FEATURE_COLS)
    df['date'] = dates
    df['pm25_target'] = pm25.round(2)
    df['aqi_target']  = aqi.round(1)
    print(f'✓ Synthetic dataset: {df.shape}')

feat_avail = [f for f in FEATURE_COLS if f in df.columns]
tgt_avail  = [t for t in TARGET_COLS  if t in df.columns]
print(f'Features: {len(feat_avail)} | Targets: {tgt_avail}')

train_mask = df['date'] <= '2021-12-31'
test_mask  = df['date'] >= '2022-01-01'
train_df   = df[train_mask].dropna(subset=feat_avail + tgt_avail[:1])
test_df    = df[test_mask ].dropna(subset=feat_avail + tgt_avail[:1])

if len(test_df) < 10:
    n_total = len(df.dropna(subset=feat_avail + tgt_avail[:1]))
    df_c = df.dropna(subset=feat_avail + tgt_avail[:1])
    train_df = df_c.iloc[:int(0.8*n_total)]
    test_df  = df_c.iloc[int(0.8*n_total):]

print(f'Train: {len(train_df)} rows | Test: {len(test_df)} rows')

X_train = train_df[feat_avail].values
X_test  = test_df[feat_avail].values
y_train_pm25 = train_df['pm25_target'].values
y_test_pm25  = test_df['pm25_target'].values

## 2. Train Baseline Models (RF + GBM)

Both models are trained with **temporal split** (train ≤ 2021, test ≥ 2022)
to respect the time-series structure and prevent data leakage.

In [ ]:
def eval_metrics(y_true, y_pred, name=''):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    r    = np.corrcoef(y_true, y_pred)[0, 1]
    print(f'[{name}]  RMSE={rmse:.2f}  MAE={mae:.2f}  R²={r2:.3f}  r={r:.3f}')
    return {'rmse': rmse, 'mae': mae, 'r2': r2, 'pearson_r': r}

models = {
    'RandomForest': RandomForestRegressor(
        n_estimators=100, max_depth=12, random_state=42, n_jobs=-1
    ),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42
    ),
}

results  = {}
preds    = {}
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train_pm25)
    y_pred = np.clip(model.predict(X_test), 0, None)
    preds[name] = y_pred
    elapsed = time.time() - t0
    m = eval_metrics(y_test_pm25, y_pred, f'{name}/pm25')
    m['elapsed_s'] = round(elapsed, 1)
    results[name] = m

print('\nTraining complete!')

## 3. Predicted vs Observed Scatter Plots

Points along the 1:1 line indicate perfect prediction. Systematic bias
shows as offset, variance shows as spread.

In [ ]:
fig, axes = plt.subplots(1, len(preds), figsize=(7 * len(preds), 6))
if len(preds) == 1:
    axes = [axes]

for ax, (name, y_pred) in zip(axes, preds.items()):
    m = results[name]
    lim = max(y_test_pm25.max(), y_pred.max()) * 1.05
    ax.scatter(y_test_pm25, y_pred, alpha=0.25, s=12, color='steelblue')
    ax.plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='1:1 line')
    ax.set_xlabel('Observed PM2.5 (µg m⁻³)')
    ax.set_ylabel('Predicted PM2.5 (µg m⁻³)')
    ax.set_title(
        f'{name}\nRMSE={m["rmse"]:.2f}  R²={m["r2"]:.3f}  r={m["pearson_r"]:.3f}',
        fontweight='bold',
    )
    ax.legend()
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.grid(True, alpha=0.3)

plt.suptitle('Predicted vs Observed PM2.5 — Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Feature Importances (Random Forest)

Feature importances from the Random Forest indicate which satellite/met variables
most influence PM2.5 prediction. INSAT AOD and boundary-layer height are typically
the dominant drivers.

In [ ]:
rf = models['RandomForest']
fi = pd.Series(rf.feature_importances_, index=feat_avail).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.barh(fi.index, fi.values,
               color=plt.cm.RdYlGn(fi.values / fi.values.max()),
               edgecolor='black', linewidth=0.5)
ax.set_xlabel('Importance (mean decrease in impurity)')
ax.set_title('Random Forest Feature Importances — PM2.5', fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
for bar, val in zip(bars, fi.values):
    ax.text(bar.get_width() + fi.values.max()*0.005,
            bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 5. Hyperparameter Search for Random Forest

A small `GridSearchCV` loop over `n_estimators` and `max_depth`.
In production use `--hparam_search` flag with `baseline_ml.py` for the full grid.

In [ ]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [8, 12, None],
}

print('Running GridSearchCV… (this may take ~60s)')
gs = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    verbose=1,
    refit=True,
)
gs.fit(X_train, y_train_pm25)

best_rf = gs.best_estimator_
best_pred = np.clip(best_rf.predict(X_test), 0, None)

print(f'\nBest params: {gs.best_params_}')
eval_metrics(y_test_pm25, best_pred, 'BestRF/pm25')

# Visualise CV results
cv_df = pd.DataFrame(gs.cv_results_)
pivot_cv = cv_df.pivot_table(
    index='param_n_estimators', columns='param_max_depth',
    values='mean_test_score',
)
print('\nCV RMSE (negative score) by n_estimators × max_depth:')
print((-pivot_cv).round(2))

## 6. CNN-LSTM Training (Synthetic Grid)

We train a small CNN-LSTM on a synthetic (T=400 days, C=13, H=15, W=15) grid
for 10 epochs to demonstrate the training loop.  Switch to real grid data
by setting `USE_REAL_DATA = True` and ensuring `grid_daily_features.csv` exists.

In [ ]:
USE_REAL_DATA = False   # flip to True once grid_daily_features.csv is available

N_DAYS   = 400
IN_CH    = len(feat_avail)
IMG_SIZE = 15
SEQ_LEN  = 7

rng_g = np.random.default_rng(1)
X_grid = rng_g.standard_normal((N_DAYS, IN_CH, IMG_SIZE, IMG_SIZE)).astype(np.float32)
y_grid = np.abs(rng_g.standard_normal((N_DAYS, IMG_SIZE, IMG_SIZE))).astype(np.float32) * 50

split_train = int(N_DAYS * 0.75)
split_val   = int(N_DAYS * 0.85)

train_ds = AQIDataset(X_grid[:split_train],        y_grid[:split_train],        SEQ_LEN)
val_ds   = AQIDataset(X_grid[split_train:split_val], y_grid[split_train:split_val], SEQ_LEN)
test_ds  = AQIDataset(X_grid[split_val:],          y_grid[split_val:],          SEQ_LEN)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=16, shuffle=False)
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=16, shuffle=False)

model = CNNLSTM(
    in_channels=IN_CH, img_h=IMG_SIZE, img_w=IMG_SIZE,
    cnn_filters=(32, 64), lstm_hidden=64, lstm_layers=1, dropout=0.2, fc_hidden=32,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'CNN-LSTM parameters: {n_params:,}')

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
criterion = nn.MSELoss()

history = {'train': [], 'val': []}
EPOCHS = 10

for epoch in range(1, EPOCHS + 1):
    tr = train_epoch(model, train_loader, optimizer, criterion, device)
    vl = val_epoch(model,  val_loader,   criterion, device)
    scheduler.step(vl)
    history['train'].append(tr)
    history['val'].append(vl)
    print(f'Epoch {epoch:2d}/{EPOCHS}  train={tr:.4f}  val={vl:.4f}  '
          f'lr={optimizer.param_groups[0]["lr"]:.2e}')

print('\nTraining complete!')

## 7. Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
epochs = range(1, len(history['train']) + 1)
ax.plot(epochs, history['train'], 'o-', color='steelblue', linewidth=2,
        markersize=5, label='Train loss (MSE)')
ax.plot(epochs, history['val'],   's--', color='darkorange', linewidth=2,
        markersize=5, label='Val loss (MSE)')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('CNN-LSTM Training & Validation Loss Curves', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

improvement = (history['train'][0] - history['train'][-1]) / history['train'][0] * 100
print(f'Training loss improved by {improvement:.1f}% over {EPOCHS} epochs')

## 8. CNN-LSTM Test Evaluation

We evaluate on the held-out test grid, then compare to baseline models.

In [ ]:
model.eval()
all_preds, all_trues = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        out = model(xb.to(device)).cpu().numpy().ravel()
        all_preds.extend(out)
        all_trues.extend(yb.numpy().ravel())

cnn_pred = np.clip(np.array(all_preds), 0, None)
cnn_true = np.array(all_trues)
m_cnn = eval_metrics(cnn_true, cnn_pred, 'CNN-LSTM/pm25')

# Comparison table
print('\n=== Model Comparison ===')
print(f'{"Model":<20} {"RMSE":>8} {"MAE":>8} {"R2":>8} {"r":>8}')
print('-' * 56)
for name, m in results.items():
    print(f'{name:<20} {m["rmse"]:8.3f} {m["mae"]:8.3f} {m["r2"]:8.4f} {m["pearson_r"]:8.4f}')
print(f'{"CNN-LSTM":<20} {m_cnn["rmse"]:8.3f} {m_cnn["mae"]:8.3f} {m_cnn["r2"]:8.4f} {m_cnn["pearson_r"]:8.4f}')

# Scatter
fig, ax = plt.subplots(figsize=(6, 6))
lim = max(cnn_true.max(), cnn_pred.max()) * 1.05
ax.scatter(cnn_true, cnn_pred, alpha=0.2, s=8, color='tomato')
ax.plot([0, lim], [0, lim], 'b--', linewidth=1.5, label='1:1 line')
ax.set_xlabel('Observed PM2.5 (µg m⁻³)')
ax.set_ylabel('CNN-LSTM Predicted PM2.5 (µg m⁻³)')
ax.set_title(
    f'CNN-LSTM | RMSE={m_cnn["rmse"]:.2f}  R²={m_cnn["r2"]:.3f}  r={m_cnn["pearson_r"]:.3f}',
    fontweight='bold',
)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. CNN-LSTM Hyperparameter Sweep (LR × Dropout)

A quick manual loop over learning rate and dropout to see their effect.
Use `python -m src.models.train_aqi --hparam_sweep` for a saved CSV output.

In [ ]:
sweep_results = []
for lr in [1e-3, 5e-4]:
    for dropout in [0.2, 0.35]:
        m_sweep = CNNLSTM(
            in_channels=IN_CH, img_h=IMG_SIZE, img_w=IMG_SIZE,
            cnn_filters=(32, 64), lstm_hidden=64, lstm_layers=1,
            dropout=dropout, fc_hidden=32,
        ).to(device)
        opt_sw = torch.optim.Adam(m_sweep.parameters(), lr=lr)
        best_vl = float('inf')
        for _ in range(6):
            train_epoch(m_sweep, train_loader, opt_sw, criterion, device)
            vl_sw = val_epoch(m_sweep, val_loader, criterion, device)
            best_vl = min(best_vl, vl_sw)
        sweep_results.append({'lr': lr, 'dropout': dropout, 'best_val': round(best_vl, 4)})
        print(f'  lr={lr:.0e}  dropout={dropout:.2f} -> best_val={best_vl:.4f}')

sweep_df = pd.DataFrame(sweep_results).sort_values('best_val')
print('\n=== Sweep Results (sorted by val loss) ===')
print(sweep_df.to_string(index=False))